# Notebook 03 — Exploratory Data Analysis & Feature Engineering

**Predicting the Success Probability of DJ, BV, and Simon's Algorithms Under Noise**

---

## Objective
In this notebook, we explore the dataset generated in Notebook 02 and prepare the features for machine learning.

### What we'll do:
- Load and inspect the dataset
- Visualize distributions and key trends
- Analyze correlations between features
- Engineer features (one-hot encoding) for the ML pipeline
- Identify initial patterns before formal modeling

## 3.1 Load Dataset

In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.feature_engineering import load_dataset, encode_features, get_dataset_summary

sns.set_theme(style='whitegrid', palette='deep')
%matplotlib inline

# Load the dataset
data_path = os.path.join(project_root, 'data', 'results.csv')
df = load_dataset(data_path)

## 3.2 Dataset Overview

In [ ]:
# Comprehensive summary
get_dataset_summary(df)

In [ ]:
# Data types and info
print('Data types:')
print(df.dtypes)
print(f'\nMemory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print(f'\nMissing values per column:')
print(df.isnull().sum())

In [ ]:
# First few rows
df.head(10)

In [ ]:
# Statistical summary of numeric columns
df.describe().round(4)

## 3.3 Distribution Plots

In [ ]:
# Distribution of success probability
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Overall distribution
axes[0].hist(df['success_probability'], bins=50, color='#4C72B0', edgecolor='white',
             alpha=0.8)
axes[0].axvline(df['success_probability'].mean(), color='red', linestyle='--',
                linewidth=2, label=f'Mean: {df["success_probability"].mean():.3f}')
axes[0].set_xlabel('Success Probability', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Overall Distribution', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)

# By algorithm
for algo in df['algorithm'].unique():
    subset = df[df['algorithm'] == algo]
    axes[1].hist(subset['success_probability'], bins=30, alpha=0.6,
                 label=algo.replace('_', ' ').title())
axes[1].set_xlabel('Success Probability', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Distribution by Algorithm', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=9)

# By noise type
for ntype in df['noise_type'].unique():
    subset = df[df['noise_type'] == ntype]
    axes[2].hist(subset['success_probability'], bins=30, alpha=0.6,
                 label=ntype.replace('_', ' ').title())
axes[2].set_xlabel('Success Probability', fontsize=12)
axes[2].set_ylabel('Count', fontsize=12)
axes[2].set_title('Distribution by Noise Type', fontsize=14, fontweight='bold')
axes[2].legend(fontsize=9)

plt.suptitle('Success Probability Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/success_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.4 Correlation Analysis

In [ ]:
# Correlation heatmap of numeric features
numeric_cols = ['n_qubits', 'circuit_depth', 'gate_count_1q', 'gate_count_2q',
                'total_gate_count', 'noise_strength', 'success_probability']
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})

ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCorrelation with success_probability:')
print(corr_matrix['success_probability'].sort_values(ascending=False).to_string())

## 3.5 Key Trends

Let's visualize the most important relationships in the data.

In [ ]:
# Trend 1: Success probability vs Qubit Count by Algorithm
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# By algorithm
for algo in df['algorithm'].unique():
    subset = df[df['algorithm'] == algo]
    grouped = subset.groupby('n_qubits')['success_probability'].mean()
    label = algo.replace('_', ' ').title()
    axes[0].plot(grouped.index, grouped.values, marker='o', linewidth=2.5,
                 markersize=8, label=label)

axes[0].set_xlabel('Number of Qubits', fontsize=12)
axes[0].set_ylabel('Mean Success Probability', fontsize=12)
axes[0].set_title('Success vs Qubit Count (by Algorithm)', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].set_ylim(-0.05, 1.05)
axes[0].grid(True, alpha=0.3)

# By noise type
for ntype in df['noise_type'].unique():
    subset = df[df['noise_type'] == ntype]
    grouped = subset.groupby('n_qubits')['success_probability'].mean()
    label = ntype.replace('_', ' ').title()
    axes[1].plot(grouped.index, grouped.values, marker='s', linewidth=2.5,
                 markersize=8, label=label)

axes[1].set_xlabel('Number of Qubits', fontsize=12)
axes[1].set_ylabel('Mean Success Probability', fontsize=12)
axes[1].set_title('Success vs Qubit Count (by Noise Type)', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].set_ylim(-0.05, 1.05)
axes[1].grid(True, alpha=0.3)

plt.suptitle('How Qubit Count Affects Success Probability', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/success_vs_qubits.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Trend 2: Success vs Noise Strength — per algorithm, per noise type
df_noisy = df[df['noise_type'] != 'none']
algorithms = df_noisy['algorithm'].unique()

fig, axes = plt.subplots(1, len(algorithms), figsize=(6 * len(algorithms), 5))
if len(algorithms) == 1:
    axes = [axes]

for ax, algo in zip(axes, algorithms):
    subset = df_noisy[df_noisy['algorithm'] == algo]
    for ntype in subset['noise_type'].unique():
        nt_subset = subset[subset['noise_type'] == ntype]
        grouped = nt_subset.groupby('noise_strength')['success_probability'].mean()
        ax.plot(grouped.index, grouped.values, marker='o', linewidth=2,
                label=ntype.replace('_', ' ').title())

    ax.set_xlabel('Noise Strength', fontsize=11)
    ax.set_ylabel('Mean Success Probability', fontsize=11)
    ax.set_title(algo.replace('_', ' ').title(), fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)

plt.suptitle('Noise Degradation Curves Per Algorithm', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/noise_degradation_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Trend 3: Heatmap — Qubits x Noise Strength → Success Probability
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

noise_types = [nt for nt in df['noise_type'].unique() if nt != 'none']

for ax, ntype in zip(axes, noise_types):
    subset = df[df['noise_type'] == ntype]
    pivot = subset.pivot_table(
        values='success_probability',
        index='n_qubits',
        columns='noise_strength',
        aggfunc='mean'
    )
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn',
                vmin=0, vmax=1, ax=ax, linewidths=0.5,
                cbar_kws={'shrink': 0.8})
    ax.set_title(ntype.replace('_', ' ').title(), fontsize=13, fontweight='bold')
    ax.set_xlabel('Noise Strength', fontsize=11)
    ax.set_ylabel('Qubits', fontsize=11)

plt.suptitle('Success Probability Heatmaps', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/noise_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Trend 4: Circuit Depth vs Success Probability (scatter)
fig, ax = plt.subplots(figsize=(10, 6))

for algo in df['algorithm'].unique():
    subset = df[(df['algorithm'] == algo) & (df['noise_type'] != 'none')]
    ax.scatter(subset['circuit_depth'], subset['success_probability'],
               alpha=0.3, s=20, label=algo.replace('_', ' ').title())

ax.set_xlabel('Circuit Depth (Transpiled)', fontsize=12)
ax.set_ylabel('Success Probability', fontsize=12)
ax.set_title('Circuit Depth vs Success Probability', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/depth_vs_success.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.6 Feature Engineering

Now we prepare the features for machine learning:
1. **One-hot encode** categorical columns (`algorithm`, `noise_type`)
2. **Drop** non-predictive columns (`oracle_desc`)
3. **Separate** features (X) from target (y = `success_probability`)

In [ ]:
# Encode features
X, y, feature_names = encode_features(df)

print(f'Feature matrix shape: {X.shape}')
print(f'Target vector shape:  {y.shape}')
print(f'\nFeature names ({len(feature_names)}):')
for i, name in enumerate(feature_names):
    print(f'  {i+1}. {name}')

In [ ]:
# Preview the encoded feature matrix
X.head(10)

In [ ]:
# Box plot: Success probability by algorithm and noise type
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=df, x='algorithm', y='success_probability', ax=axes[0],
            palette='Set2')
axes[0].set_title('Success by Algorithm', fontsize=14, fontweight='bold')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=15)

sns.boxplot(data=df, x='noise_type', y='success_probability', ax=axes[1],
            palette='Set3')
axes[1].set_title('Success by Noise Type', fontsize=14, fontweight='bold')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('Success Probability Box Plots', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/success_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Summary

In this notebook we:

1. **Loaded** the dataset and verified its structure (no missing values, correct types)
2. **Visualized** distributions of success probability overall, by algorithm, and by noise type
3. **Analyzed** correlations — noise_strength has the strongest negative correlation with success
4. **Identified** key trends:
   - Success degrades with more qubits and stronger noise
   - Different noise types have different degradation rates
   - Simon's algorithm (2n qubits) is more noise-sensitive than DJ/BV
5. **Engineered** features: one-hot encoded categoricals, producing a clean ML-ready matrix

**Next:** In Notebook 04, we'll train our ML models on this prepared dataset.